# Hindi transcription — WhisperX (large-v3)

WhisperX = faster-whisper large-v3 + VAD-based chunking + wav2vec2 word alignment.

**VAD is ON** (it is core to how WhisperX segments audio). **Speaker diarization is OFF**
— that is the separate pyannote step, which we skip, so **no HuggingFace token is needed**.

Run the cells top to bottom (Runtime -> Run all). Use a **GPU** runtime
(Runtime -> Change runtime type -> T4 GPU).

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Install WhisperX

If you later hit a `cuDNN`/`libcudnn` error during transcription, run
`!pip install -q ctranslate2==4.4.0` and restart the runtime.

In [ ]:
!pip install -q whisperx "numpy<2.1"
print('done — if this is a fresh run, restart the runtime before continuing')

## Step 3 — Settings

In [ ]:
MODEL_SIZE = "large-v3"
LANGUAGE = "hi"
TASK = "transcribe"

BATCH_SIZE = 16        # lower to 8/4 if you hit out-of-memory
DIARIZE = False        # OFF — diarization needs a pyannote HF token; we don't use it
ALIGN = True           # wav2vec2 word-level alignment (no token needed)

# Devanagari primer nudges the model toward satsang / pravachan vocabulary.
INITIAL_PROMPT = "यह एक हिंदी सत्संग प्रवचन और भजन की रिकॉर्डिंग है।"

print(f"model={MODEL_SIZE}  lang={LANGUAGE}  diarize={DIARIZE}  align={ALIGN}")

## Step 4 — Upload your FLAC (or any audio) from your drive

In [ ]:
import os
from google.colab import files

os.makedirs("audio_in", exist_ok=True)
os.makedirs("transcripts_out", exist_ok=True)

uploaded = files.upload()  # opens a file picker for your local drive

for name, data in uploaded.items():
    dest = os.path.join("audio_in", name)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"saved -> {dest}  ({len(data)/1e6:.1f} MB)")

audio_files = sorted(os.path.join("audio_in", n) for n in os.listdir("audio_in"))
print(f"\n{len(audio_files)} file(s) ready.")

## Step 5 — Load the model (VAD included)

`load_model` also loads the VAD model used to slice the audio into speech regions.

In [ ]:
import torch
import whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
if device == "cpu":
    print("WARNING: no GPU detected — this will be very slow. Switch runtime to GPU.")

print(f"Loading WhisperX {MODEL_SIZE} on {device} ({compute_type}) ...")
model = whisperx.load_model(
    MODEL_SIZE,
    device,
    compute_type=compute_type,
    language=LANGUAGE,
    asr_options={"initial_prompt": INITIAL_PROMPT},
)
print("Model + VAD ready.")

## Step 6 — Transcribe (+ word alignment), write .txt / .srt / .vtt

In [ ]:
import time
from pathlib import Path

def fmt_ts(seconds, sep=","):
    seconds = max(0.0, float(seconds or 0.0))
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

align_model = None
align_meta = None

for path in audio_files:
    stem = Path(path).stem
    print(f"\n=== {stem} ===")
    t0 = time.time()

    audio = whisperx.load_audio(path)
    print(f"  loaded {len(audio)/16000:.0f}s of audio")

    # VAD-chunked transcription
    result = model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)

    # Optional word-level alignment (improves timestamp accuracy). No token needed.
    if ALIGN:
        try:
            if align_model is None:
                align_model, align_meta = whisperx.load_align_model(
                    language_code=result["language"], device=device
                )
            result = whisperx.align(
                result["segments"], align_model, align_meta, audio, device,
                return_char_alignments=False,
            )
        except Exception as e:
            print(f"  alignment skipped ({type(e).__name__}: {e})")

    segments = result.get("segments", [])
    full_text = " ".join(s["text"].strip() for s in segments).strip()

    srt_lines, vtt_lines = [], ["WEBVTT", ""]
    for i, s in enumerate(segments, 1):
        start, end = s.get("start", 0.0), s.get("end", 0.0)
        text = s["text"].strip()
        if not text:
            continue
        srt_lines.append(f"{i}\n{fmt_ts(start)} --> {fmt_ts(end)}\n{text}\n")
        vtt_lines.append(f"{fmt_ts(start, '.')} --> {fmt_ts(end, '.')}\n{text}\n")

    Path(f"transcripts_out/{stem}.txt").write_text(full_text, encoding="utf-8")
    if srt_lines:
        Path(f"transcripts_out/{stem}.srt").write_text("\n".join(srt_lines), encoding="utf-8")
        Path(f"transcripts_out/{stem}.vtt").write_text("\n".join(vtt_lines), encoding="utf-8")
    print(f"  done in {time.time()-t0:.0f}s -> transcripts_out/{stem}.txt (+ .srt/.vtt)")

print("\nAll files transcribed.")

## Step 7 — Preview

In [ ]:
from pathlib import Path
txts = sorted(Path("transcripts_out").glob("*.txt"))
if txts:
    print(f"--- {txts[0].name} (first 1500 chars) ---\n")
    print(txts[0].read_text(encoding="utf-8")[:1500])

## Step 8 — Download results

In [ ]:
from google.colab import files
!zip -r -q transcripts.zip transcripts_out
files.download("transcripts.zip")